# Week 1 — EDGAR Data Pipeline
## Reputational Risk Topic Modelling Project

**Goal:** Pull 10-K annual reports from the SEC EDGAR database and extract the **Item 1A (Risk Factors)** section for S&P 500 companies.

**Pipeline steps:**
1. Install & import libraries
2. Fetch S&P 500 company tickers
3. Map tickers → CIK numbers (EDGAR company ID)
4. Fetch list of 10-K filings per company
5. Download and parse each 10-K document
6. Extract Item 1A text
7. Save results to disk

## 1. Install Dependencies

In [1]:
# Run this cell once to install required libraries
# Comment out after first run
import subprocess
import sys

# Upgrade pip and install dependencies
subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "requests", "pandas", "beautifulsoup4", "lxml", "tqdm"])

print("✓ All dependencies installed successfully")

  Using cached pip-26.1.2-py3-none-any.whl.metadata (4.6 kB)
Using cached pip-26.1.2-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 25.1.1
    Uninstalling pip-25.1.1:
      Successfully uninstalled pip-25.1.1
✓ All dependencies installed successfully


## 2. Imports & Configuration

In [2]:
import os
import re
import time
import json
import requests
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm
from pathlib import Path

# ── Configuration ──────────────────────────────────────────────────────────────

# EDGAR requires a User-Agent header identifying who you are (SEC policy)
# Replace with your own name and email
HEADERS = {
    "User-Agent": "daphne s_hsueh25@stud.hwr-berlin.de",
    "Accept-Encoding": "gzip, deflate",
}

# Output directory — all data will be saved here
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
(DATA_DIR / "raw_filings").mkdir(exist_ok=True)
(DATA_DIR / "item1a").mkdir(exist_ok=True)

# Rate limiting — EDGAR allows max 10 requests/sec; we stay well under
REQUEST_DELAY = 0.15  # seconds between requests

# How many companies to process (set to None for all S&P 500)
# Use a small number like 5 first to test the pipeline
MAX_COMPANIES = 5

# Year range for filings
START_YEAR = 1995
END_YEAR   = 2024

print("Configuration ready.")
print(f"Saving data to: {DATA_DIR.resolve()}")

Configuration ready.
Saving data to: /Users/hdaphne/Desktop/bipm/NLP/Group5/data


## 3. Fetch S&P 500 Company List

We pull the S&P 500 constituent list from Wikipedia — a straightforward source that's easy to parse.

In [4]:
from io import StringIO

def get_sp500_tickers() -> pd.DataFrame:
    """
    Scrape the current S&P 500 constituent list from Wikipedia.
    Returns a DataFrame with columns: ticker, company, sector, cik
    """
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    
    # Fetch manually with a browser-like User-Agent (pd.read_html gets 403 otherwise)
    wiki_headers = {"User-Agent": "Mozilla/5.0 (compatible; research project)"}
    response = requests.get(url, headers=wiki_headers)
    response.raise_for_status()

    tables = pd.read_html(StringIO(response.text))
    df = tables[0]  # First table is the main constituent list

    df = df[["Symbol", "Security", "GICS Sector", "CIK"]].copy()
    df.columns = ["ticker", "company", "sector", "cik"]

    # Tickers like BRK.B → BRK-B for EDGAR compatibility
    df["ticker"] = df["ticker"].str.replace(".", "-", regex=False)

    # Zero-pad CIK to 10 digits (EDGAR format)
    df["cik"] = df["cik"].astype(str).str.zfill(10)

    return df


sp500 = get_sp500_tickers()

# Limit for testing — remove cap for full run
if MAX_COMPANIES:
    sp500 = sp500.head(MAX_COMPANIES)

print(f"Loaded {len(sp500)} companies")
sp500.head()

Loaded 5 companies


,ticker,company,sector,cik
0,MMM,3M,Industrials,0000066740
1,AOS,A. O. Smith,Industrials,0000091142
2,ABT,Abbott Laboratories,Health Care,0000001800
3,ABBV,AbbVie,Health Care,0001551152
4,ACN,Accenture,Information Technology,0001467373


## 4. Fetch 10-K Filing Index for Each Company

EDGAR's Submissions API returns every filing a company has ever made.
We filter for `10-K` type and keep only filings within our year range.

API endpoint: `https://data.sec.gov/submissions/CIK{cik}.json`

In [5]:
SUBMISSIONS_API = "https://data.sec.gov/submissions/CIK{cik}.json"


def get_10k_filings(cik: str, start_year: int, end_year: int) -> pd.DataFrame:
    """
    Fetch a company's 10-K filing list from the EDGAR submissions API.

    Parameters
    ----------
    cik        : 10-digit zero-padded CIK string
    start_year : earliest filing year to include
    end_year   : latest filing year to include

    Returns
    -------
    DataFrame with columns: accession_number, filing_date, primary_document
    """
    url = SUBMISSIONS_API.format(cik=cik)
    response = requests.get(url, headers=HEADERS)
    response.raise_for_status()
    data = response.json()
    time.sleep(REQUEST_DELAY)

    filings = data.get("filings", {}).get("recent", {})
    if not filings:
        return pd.DataFrame()

    df = pd.DataFrame({
        "form":             filings["form"],
        "filing_date":      filings["filingDate"],
        "accession_number": filings["accessionNumber"],
        "primary_document": filings["primaryDocument"],
    })

    # Keep only 10-K filings (excludes 10-K/A amendments — include if desired)
    df = df[df["form"] == "10-K"].copy()

    # Filter by year range
    df["year"] = pd.to_datetime(df["filing_date"]).dt.year
    df = df[(df["year"] >= start_year) & (df["year"] <= end_year)]

    return df.reset_index(drop=True)


# Test on the first company
sample = sp500.iloc[0]
sample_filings = get_10k_filings(sample["cik"], START_YEAR, END_YEAR)
print(f"{sample['company']} — {len(sample_filings)} 10-K filings found")
sample_filings.head()

3M — 6 10-K filings found


,form,filing_date,accession_number,primary_document,year
0,10-K,2024-02-07,0000066740-24-000016,mmm-20231231.htm,2024
1,10-K,2023-02-08,0000066740-23-000014,mmm-20221231.htm,2023
2,10-K,2022-02-09,0000066740-22-000010,mmm-20211231.htm,2022
3,10-K,2021-02-04,0001558370-21-000737,mmm-20201231x10k.htm,2021
4,10-K,2020-02-06,0001558370-20-000581,mmm-20191231x10k62bf35.htm,2020


## 5. Download 10-K Filing Documents

Given an accession number, we fetch the actual HTML filing document.

Filing URL format:  
`https://www.sec.gov/Archives/edgar/data/{cik}/{accession_no_dashes}/{primary_document}`

In [6]:
FILING_BASE_URL = "https://www.sec.gov/Archives/edgar/data/{cik}/{accession}/{document}"


def download_filing(cik: str, accession_number: str, primary_document: str) -> str:
    """
    Download the raw HTML/text of a 10-K filing.

    Parameters
    ----------
    cik              : numeric CIK (no leading zeros needed here)
    accession_number : e.g. '0000320193-23-000106'
    primary_document : e.g. '0000320193-23-000106.htm'

    Returns
    -------
    Raw HTML string of the filing
    """
    # Accession number in URL has dashes removed
    accession_clean = accession_number.replace("-", "")
    cik_numeric = str(int(cik))  # Remove leading zeros for the URL path

    url = FILING_BASE_URL.format(
        cik=cik_numeric,
        accession=accession_clean,
        document=primary_document,
    )

    response = requests.get(url, headers=HEADERS)
    response.raise_for_status()
    time.sleep(REQUEST_DELAY)

    return response.text


# Test download
test_row = sample_filings.iloc[0]
raw_html = download_filing(sample["cik"], test_row["accession_number"], test_row["primary_document"])
print(f"Downloaded {len(raw_html):,} characters")
print(raw_html[:500])  # Preview first 500 chars

Downloaded 4,286,095 characters
<?xml version='1.0' encoding='ASCII'?>
<html xmlns="http://www.w3.org/1999/xhtml" xmlns:iso4217="http://www.xbrl.org/2003/iso4217" xmlns:link="http://www.xbrl.org/2003/linkbase" xmlns:ecd="http://xbrl.sec.gov/ecd/2023" xmlns:xbrldi="http://xbrl.org/2006/xbrldi" xmlns:mmm="http://www.mmm.com/20231231" xmlns:stpr="http://xbrl.sec.gov/stpr/2023" xmlns:dei="http://xbrl.sec.gov/dei/2023" xmlns:ix="http://www.xbrl.org/2013/inlineXBRL" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:srt="ht


## 6. Extract Item 1A (Risk Factors) Text

10-K filings are long documents. We need to isolate the **Item 1A** section.  
We use regex to find the section boundaries, then BeautifulSoup to clean the HTML tags.

In [7]:
# Regex pattern to find the start of Item 1A and end at Item 1B (or Item 2)
# The (?i) flag makes it case-insensitive
ITEM_1A_PATTERN = re.compile(
    r'(?i)item\s*1a[\s\W]*risk\s*factors(.+?)(?=item\s*1b|item\s*2)',
    re.DOTALL
)


def extract_item_1a(html: str) -> str | None:
    """
    Extract the Item 1A (Risk Factors) section from a 10-K HTML filing.

    Strategy:
    1. Strip HTML tags with BeautifulSoup to get plain text
    2. Apply regex to find the Item 1A section boundaries
    3. Clean whitespace

    Returns
    -------
    Cleaned text string, or None if section not found
    """
    # Parse HTML and extract plain text
    soup = BeautifulSoup(html, "lxml")

    # Remove script and style noise
    for tag in soup(["script", "style", "table"]):
        tag.decompose()

    text = soup.get_text(separator=" ")

    # Collapse multiple whitespace/newlines
    text = re.sub(r'\s+', ' ', text).strip()

    # Extract Item 1A section
    match = ITEM_1A_PATTERN.search(text)
    if match:
        return match.group(1).strip()

    # Fallback: some filings use different formatting — try a looser pattern
    fallback_pattern = re.compile(
        r'(?i)item\s*1a(.{100,50000})item\s*(?:1b|2)',
        re.DOTALL
    )
    match = fallback_pattern.search(text)
    if match:
        return match.group(1).strip()

    return None  # Section not found


# Test extraction
item_1a_text = extract_item_1a(raw_html)

if item_1a_text:
    print(f"Extracted {len(item_1a_text):,} characters from Item 1A")
    print("\n--- First 1000 characters ---")
    print(item_1a_text[:1000])
else:
    print("Item 1A not found — may need to adjust the regex pattern for this filing format")

/var/folders/bm/16tt0t4j0_df5q0cr1m3cbsc0000gn/T/ipykernel_26183/2851565559.py:23: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html, "lxml")


Extracted 40,528 characters from Item 1A

--- First 1000 characters ---
,” of this document, and should be considered an integral part of Part II, Item 7, “Management’s Discussion and Analysis of Financial Condition and Results of Operations.” For additional information concerning factors that may cause actual results to vary materially from those stated in the forward-looking statements, see our reports on Form 10-K, 10-Q and 8-K filed with the SEC from time to time. Item 1A. Risk Factors Provided below is a cautionary discussion of what we believe to be the most important risk factors applicable to the Company. Discussion of these factors is incorporated by reference into and considered an integral part of Part II, Item 7, “Management’s Discussion and Analysis of Financial Condition and Results of Operations.” Risks Related to the Global Economy and External Conditions * The Company’s results are impacted by the effects of, and changes in, worldwide economic, political, regulatory, i

## 7. Full Pipeline — Loop Over All Companies and Filings

Now we run everything end-to-end:
- For each company → fetch filing list → for each filing → download → extract Item 1A → save

Results are saved as:
- `data/item1a/{cik}_{year}.txt` — one file per company per year
- `data/filing_index.csv` — master index of all filings processed

In [8]:
def run_pipeline(companies: pd.DataFrame, start_year: int, end_year: int) -> pd.DataFrame:
    """
    Full EDGAR extraction pipeline.

    For each company:
      1. Fetch 10-K filing list
      2. For each filing: download HTML, extract Item 1A, save to disk

    Returns
    -------
    DataFrame: master index of all processed filings with extraction status
    """
    records = []

    for _, company in tqdm(companies.iterrows(), total=len(companies), desc="Companies"):
        cik     = company["cik"]
        ticker  = company["ticker"]
        name    = company["company"]

        # Step 1: get list of 10-K filings
        try:
            filings = get_10k_filings(cik, start_year, end_year)
        except Exception as e:
            print(f"  ✗ Could not fetch filings for {ticker}: {e}")
            continue

        if filings.empty:
            print(f"  – No 10-K filings found for {ticker} in range")
            continue

        # Step 2: process each filing
        for _, filing in filings.iterrows():
            year             = filing["year"]
            accession_number = filing["accession_number"]
            primary_document = filing["primary_document"]
            output_path      = DATA_DIR / "item1a" / f"{cik}_{year}.txt"

            # Skip if already downloaded (allows resuming interrupted runs)
            if output_path.exists():
                records.append({
                    "ticker": ticker, "company": name, "cik": cik,
                    "year": year, "accession": accession_number,
                    "status": "skipped (already exists)", "path": str(output_path)
                })
                continue

            # Download filing
            try:
                html = download_filing(cik, accession_number, primary_document)
            except Exception as e:
                records.append({
                    "ticker": ticker, "company": name, "cik": cik,
                    "year": year, "accession": accession_number,
                    "status": f"download_failed: {e}", "path": None
                })
                continue

            # Extract Item 1A
            item_1a = extract_item_1a(html)

            if item_1a:
                output_path.write_text(item_1a, encoding="utf-8")
                status = "success"
            else:
                status = "item1a_not_found"

            records.append({
                "ticker": ticker, "company": name, "cik": cik,
                "year": year, "accession": accession_number,
                "status": status, "path": str(output_path) if item_1a else None
            })

    return pd.DataFrame(records)


# Run the pipeline
index = run_pipeline(sp500, START_YEAR, END_YEAR)

# Save master index
index_path = DATA_DIR / "filing_index.csv"
index.to_csv(index_path, index=False)

print(f"\nDone. Saved index to {index_path}")
print(index["status"].value_counts())

Companies:   0%|          | 0/5 [00:00<?, ?it/s]/var/folders/bm/16tt0t4j0_df5q0cr1m3cbsc0000gn/T/ipykernel_26183/2851565559.py:23: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html, "lxml")
Companies:  20%|██        | 1/5 [00:06<00:27,  6.90s/it]/var/folders/bm/16tt0t4j0_df5q0cr1m3cbsc0000gn/T/ipyk


Done. Saved index to data/filing_index.csv
status
success             29
item1a_not_found     2
Name: count, dtype: int64


## 8. Inspect Results

In [9]:
# Summary statistics
print("=== Pipeline Summary ===")
print(f"Total filings processed : {len(index)}")
print(f"Successful extractions  : {(index['status'] == 'success').sum()}")
print(f"Item 1A not found       : {(index['status'] == 'item1a_not_found').sum()}")
print(f"Download failures       : {index['status'].str.startswith('download').sum()}")
print(f"\nYear range covered      : {index['year'].min()} – {index['year'].max()}")
print(f"Unique companies        : {index['ticker'].nunique()}")

# Filings per year
successful = index[index["status"] == "success"]
print("\n=== Successful Extractions Per Year ===")
print(successful.groupby("year").size().sort_index().to_string())

=== Pipeline Summary ===
Total filings processed : 31
Successful extractions  : 29
Item 1A not found       : 2
Download failures       : 0

Year range covered      : 2015 – 2024
Unique companies        : 5

=== Successful Extractions Per Year ===
year
2015    1
2016    1
2017    1
2018    1
2019    3
2020    4
2021    4
2022    4
2023    5
2024    5


In [10]:
# Preview one extracted Item 1A text
sample_row = successful.iloc[0]
sample_text = Path(sample_row["path"]).read_text(encoding="utf-8")

print(f"Company : {sample_row['company']}")
print(f"Year    : {sample_row['year']}")
print(f"Length  : {len(sample_text):,} characters")
print("\n" + "─" * 60)
print(sample_text[:2000])

Company : 3M
Year    : 2024
Length  : 40,528 characters

────────────────────────────────────────────────────────────
,” of this document, and should be considered an integral part of Part II, Item 7, “Management’s Discussion and Analysis of Financial Condition and Results of Operations.” For additional information concerning factors that may cause actual results to vary materially from those stated in the forward-looking statements, see our reports on Form 10-K, 10-Q and 8-K filed with the SEC from time to time. Item 1A. Risk Factors Provided below is a cautionary discussion of what we believe to be the most important risk factors applicable to the Company. Discussion of these factors is incorporated by reference into and considered an integral part of Part II, Item 7, “Management’s Discussion and Analysis of Financial Condition and Results of Operations.” Risks Related to the Global Economy and External Conditions * The Company’s results are impacted by the effects of, and changes in

## 9. Load All Extracted Texts into a Single DataFrame

This final cell consolidates all Item 1A texts into one DataFrame — the input for Week 2 (keyword filtering & preprocessing).

In [14]:
def load_corpus(filing_index: pd.DataFrame) -> pd.DataFrame:
    """
    Load all successfully extracted Item 1A texts into a single DataFrame.

    Returns
    -------
    DataFrame with columns: ticker, company, cik, year, text
    """
    rows = []
    successful_rows = filing_index[filing_index["status"] == "success"]

    for _, row in successful_rows.iterrows():
        text = Path(row["path"]).read_text(encoding="utf-8")
        rows.append({
            "ticker":  row["ticker"],
            "company": row["company"],
            "cik":     row["cik"],
            "year":    row["year"],
            "text":    text,
        })

    return pd.DataFrame(rows)


corpus = load_corpus(index)

corpus_path = DATA_DIR / "corpus.csv"
corpus.to_csv(corpus_path, index=False)

print(f"Corpus saved to {corpus_path}")
print(f"Shape: {corpus.shape}")
corpus.head()

Corpus saved to data/corpus.csv
Shape: (29, 5)


,ticker,company,cik,year,text
0,MMM,3M,0000066740,2024,",” of this document, and should be considered ..."
1,MMM,3M,0000066740,2023,",” of this document, and should be considered ..."
2,MMM,3M,0000066740,2022,",” of this document, and should be considered ..."
3,MMM,3M,0000066740,2021,",” of this document, and should be considered ..."
4,MMM,3M,0000066740,2020,",” of this document, and should be considered ..."


In [ ]:
#in Week 2, load it with:
#corpus = pd.read_csv("data/corpus.csv")

---
## Week 1 Complete ✓

**Output files:**
- `data/item1a/{cik}_{year}.txt` — individual Item 1A sections
- `data/filing_index.csv` — master index with status per filing
- `data/corpus.parquet` — consolidated DataFrame ready for Week 2

**Next step (Week 2):** Load `corpus.parquet`, generate reputational risk keywords with an LLM, and filter/preprocess the text.